## **ReAct Agent**

In [18]:
import json
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, ToolMessage
from langchain.tools import tool
from langgraph.graph import StateGraph, START, END
from typing import TypedDict

class GraphState(TypedDict):
    messages: list

@tool
def enterprise_db(employee_name:str):
    """ 
    USE THIS TOOL WHENEVER YOU WANT TO GIVE THE USER INFORMATION ABOUT ANY EMPLOYEE
    THIS TOOL TAKES THE EMPLOYEE NAME AS INPUT AND GIVES THE EMPLOYEE INFORMATION AS 
    OUTPUT IN HUMAN READABLE FORM.
    EG: FOR EMPLOYEE_NAME = 'ADIB' IT CAN RETURN ADIB IS A 27 YEAR OLD AI ENGINEER.
    """

    fake_data = {
        'adib': {
            'designation': 'Ai Engineer',
            'age': '27 years',
            'hobby': 'Playing football and basketball'
        },
        'zaid': {
            'designation': 'Robotics Engineer',
            'age': '22 years',
            'hobby': 'Likes building stuff'
        }
    }

    return fake_data.get(employee_name.lower(), 'No such user found in DB')

def llm_node(state: GraphState)->GraphState:
    messages = state['messages']
    prompt_tempelate = ChatPromptTemplate.from_messages(
        {
            ('user', '{messages}')
        }
    )    
    chain = prompt_tempelate | llm
    response = chain.invoke(
        {
            'messages': messages
        }
    )
    state['messages'] += [response]
    return state 

def tool_node(state: GraphState)->GraphState:
    messages = state['messages']
    tool_calls = messages[-1].tool_calls

    for tool_call in tool_calls:
        args = tool_call['args']
        tool_name = tool_call['name']
        tool_result = tool_registry[tool_name].invoke(args)
        state['messages'] += [ToolMessage(content=tool_result, tool_call_id=tool_call['id'])]
    
    return state

def if_tool_call(state: GraphState)->GraphState:
    last_message = state['messages'][-1]

    if hasattr(last_message, 'tool_calls') and last_message.tool_calls:
        return 'tool_node'
    return END

toolkit = [enterprise_db]
llm = ChatOpenAI(model='gpt-5', temperature=0).bind_tools(toolkit)
tool_registry = {
    'enterprise_db': enterprise_db
}

my_react_graph = StateGraph(GraphState)
my_react_graph.add_node('llm_node', llm_node)
my_react_graph.add_node('tool_node', tool_node)
my_react_graph.add_edge(START, 'llm_node')
my_react_graph.add_conditional_edges('llm_node', if_tool_call, {'tool_node': 'tool_node', END: END})
my_react_graph.add_edge('tool_node', 'llm_node')

my_react_graph_compiled = my_react_graph.compile()
# my_react_graph_compiled.invoke(
#     {
#         'messages': [
#             SystemMessage(content='You are helpful assistant, who does what it is told to do, you can use the tools if you want'),
#             HumanMessage(content='Give me some info about Adib')
#         ]
#     }
# )

for chunk in my_react_graph_compiled.stream(
    {
        'messages': [
            SystemMessage(content='You are helpful assistant, who does what it is told to do, you can use the tools if you want'),
            HumanMessage(content='Give me some info about Adib and Zaid')
        ]
    },
    stream_mode="updates"
):
    print(chunk)


{'llm_node': {'messages': [SystemMessage(content='You are helpful assistant, who does what it is told to do, you can use the tools if you want', additional_kwargs={}, response_metadata={}), HumanMessage(content='Give me some info about Adib and Zaid', additional_kwargs={}, response_metadata={}), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 377, 'prompt_tokens': 248, 'total_tokens': 625, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 320, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-5-2025-08-07', 'system_fingerprint': None, 'id': 'chatcmpl-DUCGLZ5FqdTZwfHcz9FvCGcIkrjQq', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019d872c-bb04-7b52-808d-1c4e998531fc-0', tool_calls=[{'name': 'enterprise_db', 'args': {'employee_name': 